# 02 - Préparation horaire (Individual Household Electric Power Consumption)

**Objectif :** passer du pas de 1 minute au pas de 1 heure et produire un fichier au format commun du projet (`timestamp`, `power_kW`).

Étapes :
1. Charger le dataset nettoyé
2. Agréger à l'heure (en gardant les heures trop incomplètes comme manquantes)
3. Traiter les trous : interpoler les petits, laisser les gros
4. Vérifier avec des graphes (horaire, journalier, mensuel)
5. Sauvegarder dans `01_Dataset/Processed/household_hourly.csv.gz`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_PATH = "../../01_Dataset/Processed/household_power_consumption_final.csv.gz"
OUTPUT_PATH = "../../01_Dataset/Processed/household_hourly.csv.gz"

# Paramètres de nettoyage (modifiables)
MIN_MINUTES_PER_HOUR = 45   # une heure est valide si au moins 45 minutes sur 60 sont mesurées
MAX_GAP_HOURS = 3           # on interpole seulement les trous de 3 heures ou moins

## 1. Chargement

In [ ]:
df = pd.read_csv(INPUT_PATH, compression="gzip", parse_dates=["Datetime"])

# Date et Time sont redondantes avec Datetime
df = df.drop(columns=["Date", "Time"]).set_index("Datetime").sort_index()

print("Shape :", df.shape)
print("Période :", df.index.min(), "->", df.index.max())
df.head()

## 2. Agrégation à l'heure

- `Global_active_power` est en **kW** : la moyenne sur une heure donne directement des **kWh** pour cette heure.
- Les sous-compteurs sont en **Wh par minute** : moyenne par minute x 60 = **Wh par heure**.
- Une heure avec moins de `MIN_MINUTES_PER_HOUR` minutes mesurées est mise à `NaN` (on ne se base pas sur trop peu de mesures).

In [ ]:
hourly = pd.DataFrame({
    "power_kW":        df["Global_active_power"].resample("h").mean(),
    "reactive_kW":     df["Global_reactive_power"].resample("h").mean(),
    "voltage_V":       df["Voltage"].resample("h").mean(),
    "intensity_A":     df["Global_intensity"].resample("h").mean(),
    "sub_metering_1_Wh": df["Sub_metering_1"].resample("h").mean() * 60,
    "sub_metering_2_Wh": df["Sub_metering_2"].resample("h").mean() * 60,
    "sub_metering_3_Wh": df["Sub_metering_3"].resample("h").mean() * 60,
})

# Nombre de minutes réellement mesurées dans chaque heure
minutes_count = df["Global_active_power"].resample("h").count()

# Heures trop incomplètes -> NaN
too_few = minutes_count < MIN_MINUTES_PER_HOUR
hourly.loc[too_few, :] = np.nan

print("Nombre d'heures        :", len(hourly))
print("Heures invalides (NaN) :", hourly["power_kW"].isna().sum())
print("  dont partielles (1 à 44 minutes) :", ((minutes_count > 0) & too_few).sum())
print("  dont totalement vides            :", (minutes_count == 0).sum())

## 3. Traitement des trous

Les trous sont de deux types :
- **petits** (quelques heures) : on les interpole linéairement, c'est une approximation raisonnable ;
- **gros** (jusqu'à 5 jours) : on **ne les invente pas**, on les laisse vides.

La colonne `is_interpolated` permet de savoir quelles valeurs ont été estimées.

In [ ]:
# Longueur de chaque série de NaN consécutifs
is_nan = hourly["power_kW"].isna()
gap_id = (is_nan != is_nan.shift()).cumsum()
gap_len = is_nan.groupby(gap_id).transform("sum").where(is_nan, 0)

gaps = (
    hourly.index.to_series()[is_nan]
    .groupby(gap_id[is_nan])
    .agg(start="first", end="last", hours="count")
    .sort_values("hours", ascending=False)
)
print("Nombre de trous :", len(gaps))
print("Trous <= %d h   :" % MAX_GAP_HOURS, (gaps["hours"] <= MAX_GAP_HOURS).sum())
print("Trous  > %d h   :" % MAX_GAP_HOURS, (gaps["hours"] > MAX_GAP_HOURS).sum())
gaps.head(10)

In [ ]:
hourly["is_interpolated"] = False

# Interpolation uniquement pour les petits trous
small_gap = is_nan & (gap_len <= MAX_GAP_HOURS)
interpolated = hourly.drop(columns="is_interpolated").interpolate(
    method="time", limit_area="inside"
)

cols = [c for c in hourly.columns if c != "is_interpolated"]
hourly.loc[small_gap, cols] = interpolated.loc[small_gap, cols]
hourly.loc[small_gap, "is_interpolated"] = True

print("Heures interpolées            :", int(hourly["is_interpolated"].sum()))
print("Heures restant manquantes     :", int(hourly["power_kW"].isna().sum()))

## 4. Vérification visuelle

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(hourly.index, hourly["power_kW"], linewidth=0.4)
ax.set_title("Consommation horaire (kW) - les trous restants apparaissent comme des coupures")
ax.set_xlabel("Date")
ax.set_ylabel("Puissance moyenne (kW)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Énergie journalière et mensuelle (kWh) - uniquement sur les jours / mois complets
daily_kwh = hourly["power_kW"].resample("D").sum(min_count=24)

monthly_kwh = hourly["power_kW"].resample("MS").sum(min_count=1)
hours_in_month = hourly["power_kW"].resample("MS").count()
expected_hours = hourly["power_kW"].resample("MS").size()
monthly_kwh = monthly_kwh.where(hours_in_month == expected_hours)   # mois incomplets exclus

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(daily_kwh.index, daily_kwh.values, linewidth=0.6)
axes[0].set_title("Énergie journalière (kWh)")
axes[0].grid(True, alpha=0.3)

axes[1].bar(monthly_kwh.index, monthly_kwh.values, width=25)
axes[1].set_title("Énergie mensuelle (kWh) - mois complets seulement")
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Profil moyen par heure de la journée
profile = hourly.groupby(hourly.index.hour)["power_kW"].mean()

plt.figure(figsize=(10, 4))
plt.plot(profile.index, profile.values, marker="o")
plt.xticks(range(24))
plt.title("Profil moyen par heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Puissance moyenne (kW)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Sauvegarde au format commun du projet

Colonnes obligatoires : `timestamp` et `power_kW`. Les autres colonnes sont conservées pour la prédiction et l'analyse.

In [ ]:
final = hourly.reset_index().rename(columns={"Datetime": "timestamp"})
final = final[["timestamp", "power_kW"] + [c for c in final.columns if c not in ("timestamp", "power_kW")]]

# Contrôles avant sauvegarde
assert final["timestamp"].is_monotonic_increasing
assert final["timestamp"].is_unique
assert (final["power_kW"].dropna() >= 0).all()

final.to_csv(OUTPUT_PATH, index=False, compression="gzip")

import os
print("Fichier :", OUTPUT_PATH)
print("Taille  : %.2f Mo" % (os.path.getsize(OUTPUT_PATH) / 1024 / 1024))
print("Lignes  :", len(final), "| NaN restants dans power_kW :", int(final["power_kW"].isna().sum()))
final.head()